# Week 5 Exercise -- Improved RAG for Insurellm

An enhanced RAG system that improves on the base `implementation/` in several key ways:

| Area | Base (`implementation/`) | Improved (this notebook) |
|---|---|---|
| **Chunking** | RecursiveCharacterTextSplitter (500 chars) | LLM-powered semantic chunking with headline + summary |
| **Retrieval** | Single query, top-K | Dual query (original + rewritten), merged + deduplicated |
| **Reranking** | None | LLM-based reranking of merged chunks |
| **Query** | Simple concatenation of history | LLM rewrites the question into an optimised search query |
| **Context** | Plain text | Source-attributed extracts for grounded answers |
| **Dependencies** | LangChain | Native OpenAI + ChromaDB (lighter, more transparent) |

### How to Run
1. **Cell 1** -- Run ingestion (builds the improved vector DB). Only needed once.
2. **Cell 2** -- Launch the Gradio chat UI.

In [ ]:
# ── Cell 1: Ingest the knowledge base ────────────────────────────────────────
# Run this cell ONCE to build the improved vector store.
# It uses an LLM to semantically chunk documents with headlines and summaries.

from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion

load_dotenv(override=True)

MODEL = "openai/gpt-4.1-nano"
DB_NAME = "improved_vector_db"
COLLECTION_NAME = "docs"
EMBEDDING_MODEL = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 200

openai_client = OpenAI()


# ── Data models ───────────────────────────────────────────────────────────────
class Result(BaseModel):
    page_content: str
    metadata: dict


class Chunk(BaseModel):
    headline: str = Field(
        description="A brief heading for this chunk (a few words) that captures what a user would search for"
    )
    summary: str = Field(
        description="A 2-3 sentence summary of this chunk's content, written to answer likely queries"
    )
    original_text: str = Field(
        description="The original text of this chunk from the document, exactly as written"
    )

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(
            page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,
            metadata=metadata,
        )


class Chunks(BaseModel):
    chunks: list[Chunk]


# ── Step 1: Load documents ────────────────────────────────────────────────────
def fetch_documents():
    documents = []
    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        if not folder.is_dir():
            continue
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})
    print(f"Loaded {len(documents)} documents")
    return documents


# ── Step 2: LLM-powered semantic chunking ────────────────────────────────────
def make_chunking_prompt(document):
    target_chunks = max(3, (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1)
    return f"""You split documents into overlapping chunks for a knowledge base used by a chatbot.

The document is from the company Insurellm.
Document type: {document["type"]}
Source: {document["source"]}

Split this into approximately {target_chunks} chunks (more or fewer as needed to cover the content well).
Each chunk should be self-contained enough to answer a specific question.
Include ~25% overlap between consecutive chunks so boundary content appears in multiple chunks.

For each chunk provide:
- headline: a short, searchable heading (the kind of thing a user would type)
- summary: 2-3 sentences summarising the chunk, optimised for retrieval
- original_text: the exact text from the document, unchanged

Together the chunks must cover the ENTIRE document with no content left out.

Document:

{document["text"]}

Respond with the chunks."""


def process_document(document):
    messages = [{"role": "user", "content": make_chunking_prompt(document)}]
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_chunks]


def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents, desc="Chunking documents"):
        try:
            chunks.extend(process_document(doc))
        except Exception as e:
            print(f"Error processing {doc['source']}: {e}")
    print(f"Created {len(chunks)} chunks from {len(documents)} documents")
    return chunks


# ── Step 3: Embed and store in ChromaDB ───────────────────────────────────────
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if COLLECTION_NAME in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(COLLECTION_NAME)

    texts = [chunk.page_content for chunk in chunks]

    all_vectors = []
    batch_size = 500
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        emb = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=batch).data
        all_vectors.extend([e.embedding for e in emb])

    collection = chroma.get_or_create_collection(COLLECTION_NAME)
    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]
    collection.add(ids=ids, embeddings=all_vectors, documents=texts, metadatas=metas)
    print(f"Vector store created with {collection.count()} documents")


# ── Run ingestion ─────────────────────────────────────────────────────────────
documents = fetch_documents()
all_chunks = create_chunks(documents)
create_embeddings(all_chunks)
print("Ingestion complete!")

In [ ]:
# ── Cell 2: RAG Pipeline + Gradio Chat UI ────────────────────────────────────

from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from chromadb import PersistentClient
from litellm import completion
from pydantic import BaseModel, Field
import gradio as gr

load_dotenv(override=True)

MODEL = "openai/gpt-4.1-nano"
DB_NAME = "improved_vector_db"
COLLECTION_NAME = "docs"
EMBEDDING_MODEL = "text-embedding-3-large"
RETRIEVAL_K = 20
FINAL_K = 10

openai_client = OpenAI()
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(COLLECTION_NAME)

SYSTEM_PROMPT = """You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete."""


class Result(BaseModel):
    page_content: str
    metadata: dict


class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )


# ── Query Rewriting ───────────────────────────────────────────────────────────
def rewrite_query(question, history=None):
    if history is None:
        history = []
    message = f"""You are searching a Knowledge Base about the company Insurellm.
Rewrite the user's question into a short, precise search query most likely to find relevant content.

Conversation history:
{history}

User's question:
{question}

Respond ONLY with the refined search query, nothing else."""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content.strip()


# ── Retrieval ─────────────────────────────────────────────────────────────────
def embed_query(text):
    text = str(text).strip()
    return openai_client.embeddings.create(model=EMBEDDING_MODEL, input=[text]).data[0].embedding


def fetch_context_unranked(question):
    query_vec = embed_query(question)
    results = collection.query(query_embeddings=[query_vec], n_results=RETRIEVAL_K)
    chunks = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=doc, metadata=meta))
    return chunks


def merge_chunks(chunks_a, chunks_b):
    seen = {c.page_content for c in chunks_a}
    merged = list(chunks_a)
    for chunk in chunks_b:
        if chunk.page_content not in seen:
            merged.append(chunk)
            seen.add(chunk.page_content)
    return merged


# ── Reranking ─────────────────────────────────────────────────────────────────
def rerank(question, chunks):
    system_prompt = """You are a document re-ranker.
You are given a question and a list of text chunks retrieved from a knowledge base.
Rank ALL chunks by relevance to the question, most relevant first.
Reply only with the ordered list of chunk IDs."""

    user_prompt = f"Question:\n{question}\n\nChunks:\n\n"
    for i, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {i + 1}:\n{chunk.page_content}\n\n"
    user_prompt += "Reply with the ranked chunk IDs from most to least relevant."

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    reranked = []
    for idx in order:
        if 1 <= idx <= len(chunks):
            reranked.append(chunks[idx - 1])
    return reranked


# ── Full Retrieval Pipeline ───────────────────────────────────────────────────
def fetch_context(question):
    rewritten = rewrite_query(question)
    chunks_original = fetch_context_unranked(question)
    chunks_rewritten = fetch_context_unranked(rewritten)
    merged = merge_chunks(chunks_original, chunks_rewritten)
    reranked = rerank(question, merged)
    return reranked[:FINAL_K]


# ── Answer Generation ─────────────────────────────────────────────────────────
def make_rag_messages(question, history, chunks):
    context = "\n\n".join(
        f"[Source: {chunk.metadata.get('source', 'unknown')}]\n{chunk.page_content}"
        for chunk in chunks
    )
    system = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system}] + history + [{"role": "user", "content": question}]


def answer_question(question, history=None):
    if history is None:
        history = []
    chunks = fetch_context(question)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks


# ── Gradio Chat UI ───────────────────────────────────────────────────────────
def format_context(chunks):
    html = "<h3 style='color: #00ADD8;'>Retrieved Context</h3>\n\n"
    for i, chunk in enumerate(chunks, 1):
        source = chunk.metadata.get('source', 'unknown')
        doc_type = chunk.metadata.get('type', '')
        html += (
            f"<div style='margin-bottom:12px; padding:10px; "
            f"border-left:3px solid #00ADD8; background:#f8f9fa; border-radius:4px;'>"
            f"<small style='color:#666;'>Chunk {i} | {doc_type} | {source}</small><br/>"
            f"<span style='font-size:13px;'>{chunk.page_content[:300]}{'...' if len(chunk.page_content) > 300 else ''}</span>"
            f"</div>\n"
        )
    return html


def extract_text(content):
    """Extract plain text from Gradio message content (may be str or dict)."""
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return content.get("text", content.get("content", str(content)))
    return str(content)


def chat(history):
    last_msg = history[-1]
    question = extract_text(last_msg.get("content", last_msg))

    prior = []
    for msg in history[:-1]:
        role = msg.get("role", "user")
        content = extract_text(msg.get("content", ""))
        if content:
            prior.append({"role": role, "content": content})

    answer, chunks = answer_question(question, prior)
    history.append({"role": "assistant", "content": answer})
    return history, format_context(chunks)


def add_user_message(message, history):
    return "", history + [{"role": "user", "content": message}]


with gr.Blocks(title="Insurellm RAG Assistant") as ui:

    gr.Markdown(
        "# Insurellm RAG Assistant\n"
        "Ask anything about Insurellm -- products, employees, contracts, and more.\n\n"
        "*Powered by: LLM semantic chunking, dual-query retrieval, LLM reranking, and query rewriting.*"
    )

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="Conversation", height=550)
            msg_box = gr.Textbox(
                placeholder="Ask a question about Insurellm...",
                label="",
                show_label=False,
            )

        with gr.Column(scale=2):
            context_html = gr.HTML(
                value="<p style='color:#999; text-align:center; padding:20px;'>Retrieved context will appear here after your first question.</p>"
            )

    msg_box.submit(
        add_user_message,
        inputs=[msg_box, chatbot],
        outputs=[msg_box, chatbot]
    ).then(
        chat,
        inputs=[chatbot],
        outputs=[chatbot, context_html]
    )

ui.launch(inbrowser=True, theme=gr.themes.Soft(primary_hue="teal"))